### Блок 0: Инициализация суверенной инфраструктуры
Устанавливаем оркестратор (LangGraph), локальный инференс (Ollama) и фреймворк для тестирования (`pytest`).

In [1]:
# Установка зависимостей и запуск Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!pip install -qU pydantic langchain-core langchain-ollama langgraph pytest pylint

!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time

print("Поднимаем локальный сервер инференса (Ollama)...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
with open("ollama.log", "w") as f:
    subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=os.environ)

time.sleep(7) # Ожидание инициализации демона
print("Скачиваем веса модели Qwen 2.5 Coder (3B)...")
!ollama pull qwen2.5-coder:3b
print("✅ Инфраструктура готова к работе.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:

### Блок 1: Определение состояния (State Management) и очистка кода
В мультиагентной среде мы обязаны строго типизировать состояние, которое передается между узлами. Для этого мы используем `TypedDict`. Также мы пишем утилиту для извлечения чистого кода из ответов LLM (борьба с markdown-разметкой).

In [2]:
# Типизация состояния и утилиты парсинга
from typing import TypedDict, Optional
import re

class PipelineState(TypedDict):
    requirement: str       # Бизнес-требование
    tests_code: str        # Код автотестов (написан человеком/QA)
    draft_code: str        # Код, сгенерированный агентом
    lint_error: str        # Ошибка статического анализа (если есть)
    test_error: str        # Traceback от Pytest (если есть)
    iterations: int        # Счетчик цикла (защита от зацикливания)
    status: str            # Текущий статус пайплайна

def extract_python_code(llm_response: str) -> str:
    """
    Аппаратный фильтр: извлекает чистый Python код из ответа модели,
    отсекая маркдаун (```python ... ```) и текстовые рассуждения.
    """
    match = re.search(r"```python\n(.*?)\n```", llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return llm_response.strip()

print("✅ State Management и фильтры инициализированы.")

✅ State Management и фильтры инициализированы.


### Блок 2: Изолированная песочница и Статический анализ (Quality Gates)
Мы не запускаем сгенерированный код в основном процессе. Мы симулируем CI-раннер, сохраняя код во временный файл и запуская `pytest` через подпроцесс. Это аналог эфемерного Docker-контейнера.

In [3]:
# Движок тестирования и Линтинга
def run_quality_gates(code: str, tests: str) -> tuple[bool, str, str]:
    """
    Сохраняет код и тесты, прогоняет линтер (Syntax) и Pytest (Logic).
    Возвращает: (is_success, lint_error, traceback)
    """
    # 1. Собираем единый файл для тестирования
    full_code = code + "\n\n" + tests
    with open("temp_exec.py", "w", encoding="utf-8") as f:
        f.write(full_code)

    # 2. Статический анализ (Fast Fail) - проверяем только базовый синтаксис
    try:
        compile(full_code, "temp_exec.py", "exec")
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg} at line {e.lineno}", ""

    # 3. Запуск Pytest (Логическое тестирование)
    # Используем timeout=10 для защиты от бесконечных циклов (while True) в коде ИИ
    result = subprocess.run(["pytest", "temp_exec.py", "-v", "--tb=short"],
        capture_output=True,
        text=True,
        timeout=10
    )

    if result.returncode == 0:
        return True, "", "" # Все тесты зеленые
    else:
        # Извлекаем Traceback для петли обратной связи
        return False, "", result.stdout

print("✅ Движок CI/CD (Quality Gates & Pytest Runner) готов.")

✅ Движок CI/CD (Quality Gates & Pytest Runner) готов.


*Архитектурный комментарий:* Обратите внимание на `timeout=10` в `subprocess.run`. Если агент напишет `while True: pass`, без этого таймаута пайплайн зависнет навсегда. Это базовый стандарт защиты (Sandboxing).

### Блок 3: Узлы графа
Мы создаем два узла. `coder_node` отвечает за генерацию кода на основе требований и ошибок. `ci_runner_node` выполняет физический прогон тестов и обновляет состояние графа.

In [4]:
# Определение узлов графа
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# Инициализируем локальную модель
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.1)

def coder_node(state: PipelineState) -> PipelineState:
    print(f"\n[Агент-Кодер] Итерация {state['iterations']}. Генерация кода...")

    # Формируем контекст с учетом прошлых ошибок (Traceback Loop)
    prompt = f"""Ты - Senior Python Developer. Твоя задача - написать код, который пройдет автотесты.
    ТРЕБОВАНИЕ: {state['requirement']}

    ТЕСТЫ, КОТОРЫЕ ДОЛЖНЫ ПРОЙТИ:
    ```python
    {state['tests_code']}
    ```
    """

    # Если это не первая итерация, добавляем лог ошибок
    if state['iterations'] > 1:
        prompt += f"\n\nТВОЙ ПРЕДЫДУЩИЙ КОД:\n```python\n{state['draft_code']}\n```"
        if state['lint_error']:
            prompt += f"\n\nОШИБКА СИНТАКСИСА:\n{state['lint_error']}"
        if state['test_error']:
            prompt += f"\n\nОШИБКА PYTEST (TRACEBACK):\n{state['test_error']}"
        prompt += "\nИСПРАВЬ КОД НА ОСНОВЕ ЭТИХ ОШИБОК. ВЕРНИ ТОЛЬКО ИСПРАВЛЕННЫЙ КОД."

    response = llm.invoke([HumanMessage(content=prompt)])
    clean_code = extract_python_code(response.content)

    return {
        "draft_code": clean_code,
        "iterations": state["iterations"] + 1,
        "status": "code_generated"
    }

def ci_runner_node(state: PipelineState) -> PipelineState:
    print("[CI-Runner] Запуск тестов и линтеров...")
    is_success, lint_err, test_err = run_quality_gates(state["draft_code"], state["tests_code"])

    if is_success:
        print("✅ [CI-Runner] ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!")
        return {"lint_error": "", "test_error": "", "status": "success"}
    else:
        print("❌ [CI-Runner] ТЕСТЫ УПАЛИ. Сбор Traceback...")
        return {"lint_error": lint_err, "test_error": test_err, "status": "failed"}

print("✅ Узлы Coder и CI-Runner инициализированы.")

✅ Узлы Coder и CI-Runner инициализированы.


### Блок 4: Оркестрация графа и Предохранитель (Circuit Breaker)
Собираем узлы в единый конвейер. Главная логика здесь — функция условного перехода (Conditional Edge), которая принимает решение: завершить работу, отправить на доработку или аппаратно прервать цикл при превышении лимита.

In [5]:
# Сборка графа StateGraph
from langgraph.graph import StateGraph, START, END

# Настраиваем жесткий лимит итераций (защита от выгорания токенов)
MAX_RETRIES = 4

def router_logic(state: PipelineState) -> str:
    if state["status"] == "success":
        return "end"

    if state["iterations"] >= MAX_RETRIES:
        print(f"🛑 [Circuit Breaker] Превышен лимит итераций ({MAX_RETRIES}). Аварийная остановка пайплайна.")
        return "end"

    return "retry"

# Строим направленный граф
workflow = StateGraph(PipelineState)

workflow.add_node("coder", coder_node)
workflow.add_node("ci_runner", ci_runner_node)

workflow.add_edge(START, "coder")
workflow.add_edge("coder", "ci_runner")

# Условный переход (Conditional Edge) после прогона тестов
workflow.add_conditional_edges(
    "ci_runner",
    router_logic,
    {
        "end": END,
        "retry": "coder" # Возврат к кодеру с логом ошибки
    }
)

ci_pipeline = workflow.compile()
print("✅ AI-TDD Пайплайн успешно скомпилирован.")

✅ AI-TDD Пайплайн успешно скомпилирован.


### Блок 5: Запуск боевого сценария
Задаем бизнесовое требование и жесткий набор тестов (Contract). Запускаем конвейер и наблюдаем за магией автономного исправления ошибок.

In [6]:
# Ячейка 6: Запуск пайплайна
# 1. Формализуем задачу (Spec)
requirement = """
Напиши функцию `calculate_discount(price: float, discount_percent: float) -> float`.
Функция должна применять скидку к цене.
ИНВАРИАНТ: Если discount_percent меньше 0 или больше 100, функция ОБЯЗАНА выбросить ValueError.
Возвращаемое значение должно быть округлено до 2 знаков после запятой.
"""

# 2. Пишем тесты (TDD Contract) - в проде их пишет Агент-QA или Senior Инженер
tests_code = """
import pytest

def test_normal_discount():
    assert calculate_discount(100.0, 20.0) == 80.0
    assert calculate_discount(50.0, 10.0) == 45.0

def test_rounding():
    assert calculate_discount(100.0, 33.333) == 66.67

def test_invalid_discount_low():
    with pytest.raises(ValueError):
        calculate_discount(100.0, -5.0)

def test_invalid_discount_high():
    with pytest.raises(ValueError):
        calculate_discount(100.0, 150.0)
"""

# 3. Инициализируем стартовое состояние
initial_state = {
    "requirement": requirement,
    "tests_code": tests_code,
    "draft_code": "",
    "lint_error": "",
    "test_error": "",
    "iterations": 0,
    "status": "started"
}

print("Запуск автономного CI/CD конвейера...\n" + "="*50)

# Запускаем граф
final_state = ci_pipeline.invoke(initial_state)

print("="*50)
if final_state["status"] == "success":
    print("РЕЛИЗ УСПЕШЕН. Финальный код, прошедший Quality Gates:")
    print("--------------------------------------------------")
    print(final_state["draft_code"])
    print("--------------------------------------------------")
else:
    print("❌ РЕЛИЗ ПРОВАЛЕН. Агент не смог исправить код за отведенное число итераций.")

Запуск автономного CI/CD конвейера...

[Агент-Кодер] Итерация 0. Генерация кода...
[CI-Runner] Запуск тестов и линтеров...
✅ [CI-Runner] ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!
РЕЛИЗ УСПЕШЕН. Финальный код, прошедший Quality Gates:
--------------------------------------------------
import pytest

def calculate_discount(price: float, discount_percent: float) -> float:
    if discount_percent < 0 or discount_percent > 100:
        raise ValueError("Discount percentage must be between 0 and 100.")
    
    discounted_price = price * (1 - discount_percent / 100)
    return round(discounted_price, 2)

# Тесты
def test_normal_discount():
    assert calculate_discount(100.0, 20.0) == 80.0
    assert calculate_discount(50.0, 10.0) == 45.0

def test_rounding():
    assert calculate_discount(100.0, 33.333) == 66.67

def test_invalid_discount_low():
    with pytest.raises(ValueError):
        calculate_discount(100.0, -5.0)

def test_invalid_discount_high():
    with pytest.raises(ValueError):
        calcul

### Выводы:
1. Вы увидели реализацию **Traceback Loop**. Если модель на первой итерации забудет написать `if discount_percent < 0`, `pytest` упадет с ошибкой `Failed: DID NOT RAISE <class 'ValueError'>`. Скрипт перехватит этот лог и засунет его в промпт на 2-й итерации. Модель "прочитает" лог компилятора и сама добавит `raise ValueError()`.
2. Вы увидели **Сдвиг ответственности**. Мы не проверяли код модели глазами. Мы доверились математике: если тесты зеленые, код идет в релиз. Это суть парадигмы **AI-TDD**.
3. Вы увидели **Circuit Breaker** в действии. Переменная `iterations` защитила нас от бесконечного сжигания GPU-времени в случае, если модель застрянет в галлюцинациях.

Этот конвейер — точный слепок того, как устроены автономные агенты в Enterprise-системах вроде GitLab AI CI или GitHub Copilot Workspace. Выполните код, искусственно усложните тесты и посмотрите, как долго агент будет биться над решением, прежде чем система его прервет.